In [11]:
import requests
import csv
import random
import time

# CONFIG 
langs = ["enwiki", "dewiki", "hiwiki", "jawiki"]
limit_per_lang = 500   # final desired rows per language
batch_size = 50        # number of titles per POST request
target_total = limit_per_lang * len(langs)

output_file = "../data/qid.csv"
entity_url = "https://www.wikidata.org/w/api.php"
headers = {"User-Agent": "DissertationDataCollection/1.0"}

# --- STEP 1: Function to fetch random QIDs from a given Wikipedia language ---
def get_random_titles(lang_code, count):
    site = lang_code
    url = f"https://{site.replace('wiki','')}.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "list": "random",
        "rnnamespace": 0,
        "rnlimit": count,
        "format": "json"
    }
    r = requests.get(url, params=params, headers=headers)
    r.raise_for_status()
    data = r.json()
    return [page["title"].replace(" ", "_") for page in data.get("query", {}).get("random", [])]

# --- STEP 2: Function to map titles → QIDs using POST ---
def get_qids_from_titles(lang_code, titles):
    site = lang_code
    qid_map = {}
    for i in range(0, len(titles), batch_size):
        batch = titles[i:i+batch_size]
        params = {
            "action": "wbgetentities",
            "sites": site,
            "titles": "|".join(batch),
            "format": "json"
        }
        r = requests.post(entity_url, data=params, headers=headers)  # POST avoids URL length issues
        r.raise_for_status()
        entities = r.json().get("entities", {})
        for qid, entity in entities.items():
            if not qid.startswith("Q"):
                continue
            qid_map[qid] = batch[0] if batch else None  # store at least one title (not used later)
    return qid_map

# --- STEP 3: Collect QIDs for each language ---
lang_qid_maps = {}
for lang in langs:
    print(f" Fetching random articles for {lang}...")
    titles = get_random_titles(lang, limit_per_lang * 3)  # oversample for filtering later
    print(f" Mapping {len(titles)} titles to QIDs for {lang}...")
    qid_map = get_qids_from_titles(lang, titles)
    lang_qid_maps[lang] = set(qid_map.keys())
    print(f" {lang}: {len(lang_qid_maps[lang])} unique QIDs")

# --- STEP 4: Balance the QIDs across languages ---
balanced_qids = set()
lang_assignments = {}
for lang in langs:
    sample_qids = list(lang_qid_maps[lang])
    random.shuffle(sample_qids)
    sample_qids = sample_qids[:limit_per_lang]
    for qid in sample_qids:
        balanced_qids.add(qid)
        lang_assignments.setdefault(qid, set()).add(lang)

print(f"🎯 Balanced set contains {len(balanced_qids)} QIDs")

# --- STEP 5: Fetch sitelinks & labels for balanced QIDs ---
results = []
balanced_qids_list = list(balanced_qids)
for i in range(0, len(balanced_qids_list), batch_size):
    batch = balanced_qids_list[i:i+batch_size]
    params = {
        "action": "wbgetentities",
        "ids": "|".join(batch),
        "props": "sitelinks|labels",
        "format": "json"
    }
    r = requests.post(entity_url, data=params, headers=headers)
    r.raise_for_status()
    entities = r.json().get("entities", {})
    for qid, entity in entities.items():
        label_en = entity.get("labels", {}).get("en", {}).get("value", "")
        sitelinks = entity.get("sitelinks", {})
        row = [qid, label_en]
        for lang in langs:
            row.append(sitelinks.get(lang, {}).get("title", "").replace(" ", "_"))
        results.append(row)
    time.sleep(0.2)  # be nice to API

# --- STEP 6: Save final CSV ---
with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["qid", "label_en"] + [f"{lang.replace('wiki','')}_article" for lang in langs])
    writer.writerows(results)

print(f"Saved {len(results)} rows to {output_file}")


 Fetching random articles for enwiki...
 Mapping 500 titles to QIDs for enwiki...
 enwiki: 498 unique QIDs
 Fetching random articles for dewiki...
 Mapping 500 titles to QIDs for dewiki...
 dewiki: 498 unique QIDs
 Fetching random articles for hiwiki...
 Mapping 500 titles to QIDs for hiwiki...
 hiwiki: 498 unique QIDs
 Fetching random articles for jawiki...
 Mapping 500 titles to QIDs for jawiki...
 jawiki: 494 unique QIDs
🎯 Balanced set contains 1988 QIDs
Saved 1988 rows to ../data/qid.csv


In [12]:
import pandas as pd

# Load the balanced dataset
query_file = "qid.csv"  # path to your new file
df = pd.read_csv(query_file)

print("Loaded:", df.shape)
print(df.head())

# Identify article columns
article_cols = [col for col in df.columns if col.endswith("_article")]

# Melt to long format
df_melted = df.melt(
    id_vars=["qid"], 
    value_vars=article_cols, 
    var_name="language", 
    value_name="title"
)

# Drop empty titles
df_melted = df_melted.dropna(subset=["title"])
df_melted = df_melted[df_melted["title"].str.strip() != ""]

# Clean language codes (remove "_article")
df_melted["language"] = df_melted["language"].str.replace("_article", "", regex=False)

# Ensure titles have underscores instead of spaces
df_melted["title"] = df_melted["title"].str.replace(" ", "_")

# Save converted file
output_file = "../data/qid_converted.csv"
df_melted.to_csv(output_file, index=False)

print(f" Saved {output_file} with shape {df_melted.shape}")
print(df_melted.head(10))


Loaded: (1985, 6)
          qid                    label_en         en_article  \
0     Q157032                      Prilep             Prilep   
1   Q11401070                         NaN                NaN   
2   Q18047044                      APOLD1             APOLD1   
3    Q4699838           Ajjabasavanahalli  Ajjabasavanahalli   
4  Q110290659  Bohnstedt-Gymnasium Luckau                NaN   

                   de_article     hi_article  ja_article  
0                      Prilep            NaN        プリレプ  
1                         NaN            NaN  北九東部農業協同組合  
2                         NaN            NaN         NaN  
3                         NaN  अज्जबसवनहल्ली         NaN  
4  Bohnstedt-Gymnasium_Luckau            NaN         NaN  
 Saved ../data/qid_converted.csv with shape (3450, 3)
           qid language                              title
0      Q157032       en                             Prilep
2    Q18047044       en                             APOLD1
3     Q46998

In [13]:
import pandas as pd

# Load your converted file
df = pd.read_csv("qid_converted.csv")

# Check coverage bias per QID
coverage = df.groupby("qid")["language"].nunique()

print("Coverage bias distribution:")
print(coverage.value_counts(normalize=True))

# Check language frequency overall
lang_counts = df["language"].value_counts(normalize=True)

print("\nLanguage frequency distribution:")
print(lang_counts)


Coverage bias distribution:
language
1    0.534509
2    0.264484
3    0.129471
4    0.071537
Name: proportion, dtype: float64

Language frequency distribution:
language
en    0.374203
de    0.260870
ja    0.211884
hi    0.153043
Name: proportion, dtype: float64


In [1]:
import os, time, urllib.parse, requests, pickle
import pandas as pd
from collections import defaultdict

DATA_PATH = "../data"
LANGS = ["en","de","hi","ja"]
QUERY_FILE = os.path.join(DATA_PATH, "qid_converted.csv")

# polite headers
UA = "TU-Dublin-Dissertation/1.0 (contact: D24125465@mytudublin.ie)"
TIMEOUT = 30
SLEEP = 0.2  # be nice; increase if you hit rate limits

def norm_title(s):
    if s is None: return None
    return urllib.parse.unquote(str(s)).strip().replace(" ", "_")

def mw_api(lang):
    return f"https://{lang}.wikipedia.org/w/api.php"

def fetch_outlinks(lang, title):
    """Return list of article titles that 'title' links to (namespace 0)."""
    out = []
    params = {
        "action":"query","prop":"links","titles":title,
        "plnamespace":0,"pllimit":"max","format":"json"
    }
    cont = {}
    while True:
        try:
            r = requests.get(mw_api(lang), params={**params, **cont}, headers={"User-Agent": UA}, timeout=TIMEOUT)
            r.raise_for_status()
            data = r.json()
        except Exception:
            break
        pages = data.get("query",{}).get("pages",{})
        for p in pages.values():
            for link in p.get("links",[]):
                t = norm_title(link.get("title"))
                if t: out.append(t)
        cont = data.get("continue",{})
        if not cont: break
        time.sleep(SLEEP)
    return list(dict.fromkeys(out))  # dedup keep order

def fetch_inlinks(lang, title):
    """Return list of article titles that link to 'title' (namespace 0)."""
    out = []
    params = {
        "action":"query","prop":"linkshere","titles":title,
        "lhnamespace":0,"lhlimit":"max","format":"json"
    }
    cont = {}
    while True:
        try:
            r = requests.get(mw_api(lang), params={**params, **cont}, headers={"User-Agent": UA}, timeout=TIMEOUT)
            r.raise_for_status()
            data = r.json()
        except Exception:
            break
        pages = data.get("query",{}).get("pages",{})
        for p in pages.values():
            for link in p.get("linkshere",[]):
                t = norm_title(link.get("title"))
                if t: out.append(t)
        cont = data.get("continue",{})
        if not cont: break
        time.sleep(SLEEP)
    return list(dict.fromkeys(out))  # dedup keep order

# --- Load your aligned topics
qc = pd.read_csv(QUERY_FILE)
qc["title"] = qc["title"].apply(norm_title)

# Build per-language expansions
for lang in LANGS:
    lang_topics = sorted(set(qc.loc[qc["language"]==lang, "title"].dropna()))
    if not lang_topics:
        print(f"⚠️ {lang}: no topics found in query_converted.csv")
        continue

    print(f"\n🌐 {lang}: expanding {len(lang_topics)} topics to 1-hop (API) ...")
    expanded = defaultdict(set)

    for i, t in enumerate(lang_topics, 1):
        # ensure node exists
        expanded[t]

        # Outlinks: t -> neighbor
        outs = fetch_outlinks(lang, t)
        for o in outs:
            expanded[t].add(o)

        # Inlinks: src -> t
        ins = fetch_inlinks(lang, t)
        for s in ins:
            expanded[s].add(t)

        if i % 50 == 0:
            print(f"  ...{i}/{len(lang_topics)} processed")
        time.sleep(SLEEP)  # gentle throttle

    # finalize: convert sets to sorted lists
    expanded = {k: sorted(v) for k, v in expanded.items()}

    out_path = os.path.join(DATA_PATH, f"{lang}_filtered_graph_expanded.pkl")
    with open(out_path, "wb") as f:
        pickle.dump(expanded, f, protocol=pickle.HIGHEST_PROTOCOL)

    num_edges = sum(len(v) for v in expanded.values())
    print(f"✅ {lang}: saved {out_path} | nodes={len(expanded)} edges={num_edges}")



🌐 en: expanding 1291 topics to 1-hop (API) ...
  ...50/1291 processed
  ...100/1291 processed
  ...150/1291 processed
  ...200/1291 processed
  ...250/1291 processed
  ...300/1291 processed
  ...350/1291 processed
  ...400/1291 processed
  ...450/1291 processed
  ...500/1291 processed
  ...550/1291 processed
  ...600/1291 processed
  ...650/1291 processed
  ...700/1291 processed
  ...750/1291 processed
  ...800/1291 processed
  ...850/1291 processed
  ...900/1291 processed
  ...950/1291 processed
  ...1000/1291 processed
  ...1050/1291 processed
  ...1100/1291 processed
  ...1150/1291 processed
  ...1200/1291 processed
  ...1250/1291 processed
✅ en: saved ../data/en_filtered_graph_expanded.pkl | nodes=334158 edges=588918

🌐 de: expanding 900 topics to 1-hop (API) ...
  ...50/900 processed
  ...100/900 processed
  ...150/900 processed
  ...200/900 processed
  ...250/900 processed
  ...300/900 processed
  ...350/900 processed
  ...400/900 processed
  ...450/900 processed
  ...500/900 pr